# Titanic 데이터 탐색

목표: 컬럼 구조, 결측치, 타겟 분포를 확인하고 SHAP 분석에 쓸 주요 특성을 추린다.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/titanic.csv")
df.shape

(1309, 28)

In [2]:
df.head()

,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


In [3]:
df.dtypes

Passengerid      int64
Age            float64
Fare           float64
Sex              int64
sibsp            int64
zero             int64
zero.1           int64
zero.2           int64
zero.3           int64
zero.4           int64
zero.5           int64
zero.6           int64
Parch            int64
zero.7           int64
zero.8           int64
zero.9           int64
zero.10          int64
zero.11          int64
zero.12          int64
zero.13          int64
zero.14          int64
Pclass           int64
zero.15          int64
zero.16          int64
Embarked       float64
zero.17          int64
zero.18          int64
2urvived         int64
dtype: object

In [4]:
df.isna().sum()

Passengerid    0
Age            0
Fare           0
Sex            0
sibsp          0
zero           0
zero.1         0
zero.2         0
zero.3         0
zero.4         0
zero.5         0
zero.6         0
Parch          0
zero.7         0
zero.8         0
zero.9         0
zero.10        0
zero.11        0
zero.12        0
zero.13        0
zero.14        0
Pclass         0
zero.15        0
zero.16        0
Embarked       2
zero.17        0
zero.18        0
2urvived       0
dtype: int64

In [5]:
# 고유값 개수 오름차순 -> 1이면 상수 컬럼(정보 없음), 행 수와 같으면 ID성 컬럼
df.nunique().sort_values()

zero.1            1
zero.2            1
zero              1
zero.7            1
zero.6            1
zero.5            1
zero.4            1
zero.3            1
zero.9            1
zero.8            1
zero.15           1
zero.14           1
zero.13           1
zero.12           1
zero.11           1
zero.10           1
zero.18           1
zero.17           1
zero.16           1
2urvived          2
Sex               2
Pclass            3
Embarked          3
sibsp             7
Parch             8
Age              98
Fare            281
Passengerid    1309
dtype: int64

## 발견 1: 이 CSV는 표준 Titanic 데이터셋이 아니다

- 컬럼명이 일부 오염되어 있음: 타겟 컬럼명이 `Survived`가 아니라 **`2urvived`**
- 이름이 전부 `zero`인 컬럼이 다수 존재 (pandas가 읽으면서 `zero`, `zero.1`, `zero.2`, ... 로 자동 구분). 위 `nunique()` 결과에서 이 컬럼들은 고유값 1개(전부 0)로, 아무 정보가 없는 더미 컬럼임을 확인할 수 있음
- `Sex`, `Embarked` 도 이미 정수로 인코딩되어 들어있음 (원본 문자열이 아님)
- `Passengerid`는 행마다 값이 달라 ID성 컬럼

→ `common.py`의 `load_and_preprocess`는 (1) 고유값이 1개뿐인 상수 컬럼, (2) 행 수만큼 고유한 ID성 컬럼을 자동으로 제거하도록 만들어서, 컬럼명을 하드코딩하지 않고도 이 `zero*`/`Passengerid` 컬럼들이 걸러지게 했다. 타겟 컬럼명만 `train_titanic.py`에서 `"2urvived"`로 명시.

In [6]:
df["2urvived"].value_counts(normalize=True)

2urvived
0    0.738732
1    0.261268
Name: proportion, dtype: float64

In [7]:
# 상수/ID성 컬럼을 제외한 실질 컬럼만 추려서 범주형/수치형 구분
n = len(df)
useful_cols = [c for c in df.columns if c != "2urvived" and df[c].nunique() > 1 and df[c].nunique() < n]
numeric_like = [c for c in useful_cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 10]
categorical_like = [c for c in useful_cols if c not in numeric_like]
print("수치형(연속):", numeric_like)
print("범주형/이산:", categorical_like)

수치형(연속): ['Age', 'Fare']
범주형/이산: ['Sex', 'sibsp', 'Parch', 'Pclass', 'Embarked']


## 클래스 불균형

생존(`1`) 비율이 약 30%대로, 완전한 균형은 아니지만 심하게 치우치지도 않음 (RandomForest baseline에 큰 문제 없는 수준).

## SHAP 분석에 쓸 주요 특성 5~8개

1. **Sex** — 성별에 따른 생존율 차이가 Titanic에서 가장 잘 알려진 신호
2. **Pclass** — 객실 등급 (사회경제적 지위 proxy)
3. **Age** — 연령대 (어린이 우선 구조 등)
4. **Fare** — 운임 (Pclass와 상관되지만 개별 신호도 있음)
5. **sibsp** — 동승한 형제/배우자 수
6. **Parch** — 동승한 부모/자녀 수
7. **Embarked** — 승선 항구

나머지 `zero*` 더미 컬럼과 `Passengerid`는 정보가 없거나 ID성이므로 제외.